In [1]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, \
RocCurveDisplay, roc_auc_score, r2_score, mean_absolute_error, f1_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import ridge_regression, ElasticNet
from sklearn.linear_model import Ridge


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('../Cases/Glass_Identification/Glass.csv')
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,building_windows_float_processed
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,building_windows_float_processed
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,building_windows_float_processed
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,building_windows_float_processed
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,building_windows_float_processed


In [5]:
ohe = OneHotEncoder(drop = 'first', sparse_output = False)
col_trnf = ColumnTransformer([('OHE', ohe,make_column_selector(dtype_include=object) )],
                             remainder='passthrough',
                             verbose_feature_names_out=False)

col_trnf = col_trnf.set_output(transform = 'pandas')

x = col_trnf.fit_transform(x)

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=25, test_size=0.3)

In [30]:
features = [2,3,4,5]
n_est = [25, 50, 100, 150, 200]
scores = []
for f in tqdm(features):
    for n in n_est:
        rf = RandomForestRegressor(random_state = 25, max_features=f, n_estimators=n)
        rf.fit(x_train, y_train)
        y_pred = rf.predict(x_test)
        scores.append([f,n,mean_absolute_error(y_test, y_pred)])

df_scores = pd.DataFrame(data = scores, columns = ['No. of Features', 'No of Estimators', 'score'])
df_scores.sort_values('score', ascending=True)

100%|██████████| 4/4 [00:24<00:00,  6.17s/it]


,No. of Features,No of Estimators,score
6,3,50,0.049638
8,3,150,0.049925
7,3,100,0.050098
9,3,200,0.050142
11,4,50,0.050156
5,3,25,0.050166
10,4,25,0.050420
18,5,150,0.050538
13,4,150,0.050557
12,4,100,0.050577


In [31]:
features = [2,3,4,5]
n_est = [25, 50, 100, 150, 200]
scores = []
for f in tqdm(features):
    for n in n_est:
        rf = RandomForestClassifier(random_state = 25, max_features=f, n_estimators=n)
        rf.fit(x_train, y_train)
        y_pred = rf.predict(x_test)
        scores.append([f,n,f1_score(y_test, y_pred, pos_label=1)])

df_scores = pd.DataFrame(data = scores, columns = ['No. of Features', 'No of Estimators', 'score'])
df_scores.sort_values('score', ascending=False)

100%|██████████| 4/4 [00:27<00:00,  6.98s/it]


,No. of Features,No of Estimators,score
10,4,25,0.340909
11,4,50,0.285714
2,2,100,0.282051
6,3,50,0.275000
17,5,100,0.265060
19,5,200,0.258824
15,5,25,0.258065
18,5,150,0.255814
3,2,150,0.253165
0,2,25,0.250000


In [32]:
df_scores.sort_values('score', ascending=False)

,No. of Features,No of Estimators,score
10,4,25,0.340909
11,4,50,0.285714
2,2,100,0.282051
6,3,50,0.275000
17,5,100,0.265060
19,5,200,0.258824
15,5,25,0.258065
18,5,150,0.255814
3,2,150,0.253165
0,2,25,0.250000


Classification Report

In [34]:
best_model = RandomForestClassifier(random_state = 25, max_features=4, n_estimators=25)
best_model.fit(x_train, y_train)
y_pred = best_model.predict(x_test)
# scores.append([f,n,f1_score(y_test, y_pred, pos_label=1)])
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1983
           1       0.60      0.24      0.34        63

    accuracy                           0.97      2046
   macro avg       0.79      0.62      0.66      2046
weighted avg       0.96      0.97      0.97      2046



using wine dataset

In [35]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
wine = fetch_ucirepo(id=109) 
  
# data (as pandas dataframes) 
X = wine.data.features 
y = wine.data.targets 
  
# metadata 
print(wine.metadata) 
  
# variable information 
print(wine.variables) 


{'uci_id': 109, 'name': 'Wine', 'repository_url': 'https://archive.ics.uci.edu/dataset/109/wine', 'data_url': 'https://archive.ics.uci.edu/static/public/109/data.csv', 'abstract': 'Using chemical analysis to determine the origin of wines', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 178, 'num_features': 13, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1992, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C5PC7J', 'creators': ['Stefan Aeberhard', 'M. Forina'], 'intro_paper': {'ID': 246, 'type': 'NATIVE', 'title': 'Comparative analysis of statistical pattern recognition methods in high dimensional settings', 'authors': 'S. Aeberhard, D. Coomans, O. Vel', 'venue': 'Pattern Recognition', 'year': 1994, 'journal': None, 'DOI': '10.1016/0031-3203(94)90145-7', 'URL': 'https:

In [36]:
y.value_counts()

class
2        71
1        59
3        48
Name: count, dtype: int64